In [ ]:
!pip install brian2 brian2hears librosa numpy scipy matplotlib setuptools

import os
import glob
import random
import numpy as np
import tensorflow as tf
from scipy import signal

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Schakel storende waarschuwingen uit
import logging
logging.getLogger('brian2').setLevel(logging.ERROR)
import brian2
brian2.prefs.codegen.target = 'numpy'

TIME_WINDOW = 640 
DROPOUT_RATE = 0.15

pad_naar_train_val_eeg = "/content/drive/MyDrive/p&d/fase_2/data/64 Channel Biosemi unprocessed data - train/*/*/*.npz"
pad_naar_test_eeg = "/content/drive/MyDrive/p&d/fase_2/data/test_data/test_data/*/*.npz"
pad_naar_audio = "/content/drive/MyDrive/p&d/fase_2/data/preprocessed/audio/cnn/"

def process_eeg_file(npz_filename, mode='cnn'):
    data = np.load(npz_filename)
    eeg_data = data['eeg']
    fs = int(data['fs'])
    attended_wav = str(data['stimulus_attended'])
    unattended_wav = str(data['stimulus_unattended'])

    target_sr = 64
    lowcut = 1.0
    highcut = 32.0

    sos = signal.butter(N=4, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')
    eeg_filtered = signal.sosfiltfilt(sos, eeg_data, axis=0)

    from math import gcd
    g = gcd(fs, target_sr)
    eeg_downsampled = signal.resample_poly(eeg_filtered, target_sr // g, fs // g, axis=0)

    return eeg_downsampled, attended_wav, unattended_wav

def batch_equalizer(eeg, env_1, env_2, labels):
    return (np.concatenate([eeg,eeg], axis=0),
            np.concatenate([env_1, env_2], axis=0),
            np.concatenate([env_2, env_1], axis=0)), \
           np.concatenate([labels, (labels+1)%2], axis=0)

class DataGenerator:
    def __init__(self, files, audio_dir, time_window):
        self.files = files
        self.audio_dir = audio_dir
        self.time_window = time_window

        alle_audio = glob.glob(os.path.join(self.audio_dir, "**", "*.npy"), recursive=True)
        self.audio_dict = {os.path.basename(f).lower(): f for f in alle_audio}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, recording_index):
        eeg_pad = self.files[recording_index]
        eeg_processed, att_naam, unatt_naam = process_eeg_file(eeg_pad, mode='cnn')

        att_base = att_naam.replace('.wav', '').replace('.npy', '').strip()
        unatt_base = unatt_naam.replace('.wav', '').replace('.npy', '').strip()

        zoek_att = f"{att_base.lower()}_cnn.npy"
        zoek_unatt = f"{unatt_base.lower()}_cnn.npy"

        att_audio_pad = self.audio_dict.get(zoek_att)
        unatt_audio_pad = self.audio_dict.get(zoek_unatt)

        if not att_audio_pad or not unatt_audio_pad:
            raise FileNotFoundError(f"Audio niet gevonden voor {eeg_pad}")

        env1 = np.load(att_audio_pad)
        env2 = np.load(unatt_audio_pad)

        min_len = min(eeg_processed.shape[0], env1.shape[0], env2.shape[0])
        eeg_processed = eeg_processed[:min_len]
        env1 = env1[:min_len]
        env2 = env2[:min_len]

        num_windows = min_len // self.time_window
        eeg_windows = np.array(np.split(eeg_processed[:num_windows * self.time_window], num_windows))
        env1_windows = np.array(np.split(env1[:num_windows * self.time_window], num_windows))
        env2_windows = np.array(np.split(env2[:num_windows * self.time_window], num_windows))

        env1_windows = np.expand_dims(env1_windows, axis=-1)
        env2_windows = np.expand_dims(env2_windows, axis=-1)

        labels = np.ones((num_windows, 1))

        (eeg_bal, env1_bal, env2_bal), labels_bal = batch_equalizer(eeg_windows, env1_windows, env2_windows, labels)

        return (eeg_bal, env1_bal, env2_bal), labels_bal

    def __call__(self):
        for idx in range(self.__len__()):
            try:
                yield self.__getitem__(idx)
            except Exception as e:
                continue
            if idx == self.__len__() - 1:
                self.on_epoch_end()

    def on_epoch_end(self):
        np.random.shuffle(self.files)

def maak_tf_dataset(files, audio_dir, time_window):
    generator = DataGenerator(files=files, audio_dir=audio_dir, time_window=time_window)
    dataset = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            (tf.TensorSpec(shape=(None, time_window, 64), dtype=tf.float32),
             tf.TensorSpec(shape=(None, time_window, 1), dtype=tf.float32),
             tf.TensorSpec(shape=(None, time_window, 1), dtype=tf.float32)),
            tf.TensorSpec(shape=(None, 1), dtype=tf.float32)
        )
    )
    return dataset


alle_train_bestanden = glob.glob(pad_naar_train_val_eeg)
test_bestanden = glob.glob(pad_naar_test_eeg)

random.seed(42)
random.shuffle(alle_train_bestanden)

split_index = int(len(alle_train_bestanden) * 0.8)
train_files = alle_train_bestanden[:split_index]
val_files = alle_train_bestanden[split_index:]

print(f"Trainingsset:  {len(train_files)} trials")
print(f"Validatieset:  {len(val_files)} trials")
print(f" Testset: {len(test_bestanden)} trials")

train_dataset = maak_tf_dataset(train_files, pad_naar_audio, TIME_WINDOW)
val_dataset = maak_tf_dataset(val_files, pad_naar_audio, TIME_WINDOW)
test_dataset = maak_tf_dataset(test_bestanden, pad_naar_audio, TIME_WINDOW)

def bouw_dilated_model(time_window, layers=3, kernel_size=3, spatial_filters=8, dilated_filters=16, dropout_rate=0.15):
    eeg = tf.keras.layers.Input(shape=[time_window, 64], name="EEG_Input")
    env1 = tf.keras.layers.Input(shape=[time_window, 1], name="Env1_Input")
    env2 = tf.keras.layers.Input(shape=[time_window, 1], name="Env2_Input")

    env_proj_1 = env1
    env_proj_2 = env2

    eeg_proj_1 = tf.keras.layers.Conv1D(spatial_filters, kernel_size=1, name="Spatial_Conv")(eeg)

    for l in range(layers):
        eeg_proj_1 = tf.keras.layers.Conv1D(
            dilated_filters, kernel_size=kernel_size, dilation_rate=kernel_size ** l,
            strides=1, activation="relu", name=f"EEG_Dilated_{l}"
        )(eeg_proj_1)

        env_proj_layer = tf.keras.layers.Conv1D(
            dilated_filters, kernel_size=kernel_size, dilation_rate=kernel_size ** l,
            strides=1, activation="relu", name=f"Audio_Dilated_{l}"
        )
        env_proj_1 = env_proj_layer(env_proj_1)
        env_proj_2 = env_proj_layer(env_proj_2)

    cos1 = tf.keras.layers.Dot(axes=1, normalize=True, name="Cosine_Env1")([eeg_proj_1, env_proj_1])
    cos2 = tf.keras.layers.Dot(axes=1, normalize=True, name="Cosine_Env2")([eeg_proj_1, env_proj_2])

    concat = tf.keras.layers.Concatenate()([cos1, cos2])
    flat = tf.keras.layers.Flatten()(concat)

    dropout = tf.keras.layers.Dropout(dropout_rate, name=f"Dropout_{dropout_rate}")(flat)

    out1 = tf.keras.layers.Dense(1, activation="sigmoid", name="Decision_Neuron")(dropout)
    out = tf.keras.layers.Reshape([1], name="output_name")(out1)

    model = tf.keras.Model(inputs=[eeg, env1, env2], outputs=[out])
    model.compile(optimizer='adam', metrics=["accuracy"], loss="binary_crossentropy")
    return model

model = bouw_dilated_model(time_window=TIME_WINDOW, layers=3, dropout_rate=DROPOUT_RATE)



early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

pad_voor_backup = f'/content/drive/MyDrive/p&d/fase_2/dilated_3layer_BACKUP_{TIME_WINDOW}samples_dropout_{DROPOUT_RATE}.keras'

model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath=pad_voor_backup,
    monitor='val_accuracy',
    save_best_only=True, 
    verbose=1
)

geschiedenis = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=[early_stopping, model_checkpoint]
)


test_loss, test_accuracy = model.evaluate(test_dataset)
print(f"\n FINALE TEST ACCURACY: {test_accuracy * 100:.2f}%")

naam_model = f'/content/drive/MyDrive/p&d/fase_2/dilated_3layer_{TIME_WINDOW}samples_dropout_{DROPOUT_RATE}.keras'
naam_geschiedenis = f'/content/drive/MyDrive/p&d/fase_2/geschiedenis_3layer_{TIME_WINDOW}samples_dropout_{DROPOUT_RATE}.npy'

model.save(naam_model)
np.save(naam_geschiedenis, geschiedenis.history)
